In [14]:
import logging  #bilgi vermeyi sağlayan standart kütüphane bilgileri print ile vermektense logging içindeki farklı durumlara göre çıktı vermek daha doğru bir kullanımdır
from pathlib import Path
import json
import matplotlib.pyplot as plt
import torch #modelin sinir ağı bu kütüphane üzerinden çalışacak
import cv2
from transformers import AutoProcessor, AutoModelForMultimodalLM
#Processor image i modelin anlayabileceği sayısal verilere(çok boyutlu sayısal diziler) çevirir
#AutoModelForMultimodalLM → MiniCPM-V modelini yükler config dosyasına göre modeli yükler
#AutoProcessor            → Görsel + metni modele hazırlar

In [15]:
#modelin ve processorun ortama yüklenmesi 
MODEL_ID = "openbmb/MiniCPM-V-4.6-BNB"

processor = AutoProcessor.from_pretrained(MODEL_ID) #Sadece görsel ve metni ileride nasıl hazırlayacağını bilen processor nesnesini oluşturuyor.
model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        device_map="auto" #modelin nerede çalıştırılacağı otomatik belirlenecek (CPU veya GPU)
    )
model.eval() #eğitim değil de inference modunda çalıştırılacak. Katmanların çalışma davranışını inference'a uygun hale getirir.

Loading weights: 100%|██████████| 779/779 [00:02<00:00, 354.62it/s]


MiniCPMV4_6ForConditionalGeneration(
  (model): MiniCPMV4_6Model(
    (vision_tower): MiniCPMV4_6VisionModel(
      (embeddings): MiniCPMV4_6VisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4900, 1152)
      )
      (encoder): MiniCPMV4_6VisionEncoder(
        (layers): ModuleList(
          (0-26): 27 x MiniCPMV4_6VisionEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (self_attn): MiniCPMV4_6VisionAttention(
              (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affin

In [16]:
#modelin kurulumu durumu hakkında bilgilendirme
print("Model device:")
print(model.device)

print("Model class")
print(type(model))

total_params = sum(parameter.numel() for parameter in model.parameters())
print(f"Total parameters: {total_params}")

Model device:
cuda:0
Model class
<class 'transformers.models.minicpmv4_6.modeling_minicpmv4_6.MiniCPMV4_6ForConditionalGeneration'>
Total parameters: 780872944


In [17]:
#image in yüklenmesi
IMAGE_PATH = Path("test_pictures\catt.jpg")

image_bgr = cv2.imread(str(IMAGE_PATH))
if image_bgr is None:
    raise ValueError(f"Image not found at {IMAGE_PATH}")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB) #renk kanallarını değiştiriyoruz

In [18]:
#promptun düzenlenmesi
def build_prompt(user_question):
    question = f"""
Find the object requested by the user in the image.

Return only this format:

[
    {{
        "label": "object name",
        "box": [x1, y1, x2, y2]
    }}
]

User request:
{user_question}
"""
    return question

In [19]:
#chat modeli için inputu belirli bir konuşma yapısına getirilmesi
def build_messages(image_rgb, question):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_rgb
                },
                {
                    "type": "text",
                    "text": question
                }
            ]
        }
    ]

    return messages

In [20]:
def ask_model(processor, model, image_rgb, question):
    #mesajın oluşturulması
    messages = build_messages(
        image_rgb,
        question
    )

    #mesajın processor a verilmesi ve sonucunda inputun artık modelin anlayacağı bir formata dönüştürülmesi
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True, #girdileri tokenlara çevir
        add_generation_prompt=True, #özel tokenleri ekle
        return_dict=True, #girdileri dictionary formatında döndür
        return_tensors="pt", #çıktının tensor formatında olmasını sağla
    )

    inputs = inputs.to(model.device)

    #modelin cevap üretmesi
    with torch.inference_mode(): #inference modunda olduğumuz için gradient hesaplarını kapatıyor
        outputs = model.generate(
            **inputs,
            max_new_tokens=100, #cevabın uzunluğu 100 token ile sınırlandırılıyor
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:] #model çıktı üretirken çıkışın başında inputu da eklediği için onu kesip sadece modelin ürettiği kısmı alıyoruz
    #burada inputs["input_ids"] kısmı girdilerin dictionarysindeki girdi değerleirni alır
    #.shade[1] ile de tensorün boyutunu gösterir
    # : ile de tensör boyutu kadar olan kısımdaki tensörleri keser.

    #çıktı oluşturulurken girdinin hemen #modelin cevap üretmesi
    #arkasına yeni tokenler ekleniyor o yüzden cevabı oluşturuken promptu silmek gerekli

    # Modelin ürettiği cevabı token ID'lerinden string'e çevirme
    response = processor.batch_decode(
        generated_tokens,
        skip_special_tokens=True,  # <eos>, <pad>, <assistant> gibi özel tokenları gösterme
    )[0]  # Batch içindeki ilk cevabı al tek görsel tek soru olduğu için

    return response

In [21]:
#çıktının JSON formatında olup olmadığını kontrol etme ve Python nesnesine dönüştürme
def parse_model_response(response):
        try:
                detections = json.loads(response)  # JSON stringini Python nesnesine dönüştür
                return detections
        except json.JSONDecodeError:
                print("Model's answer is not JSON.")
                return []

model
→ token ID tensoru
→ decode
→ string
→ json.loads()
→ Python list/dictionary

In [22]:
def scale_box_to_pixels(box, width, height):

        x1, y1, x2, y2 = box

        x1_px = int((x1 / 1000) * width)
        y1_px = int((y1 / 1000) * height)
        x2_px = int((x2 / 1000) * width)
        y2_px = int((y2 / 1000) * height)

        return x1_px, y1_px, x2_px, y2_px 

In [ ]:
def draw_bbox(image, box, label):
    height, width = image.shape[:2]

    x1_px, y1_px, x2_px, y2_px = scale_box_to_pixels(
        box,
        width,
        height
    )

    output_image = image.copy() #buradaki image a image_bgr verilecek

    cv2.rectangle(
        output_image,
        (x1_px, y1_px),
        (x2_px, y2_px),
        (0, 255, 0),
        2
    )

    cv2.putText(
        output_image,
        label,
        (x1_px, max(y1_px - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    return output_image

In [24]:
#modelin oluşturduğu resmi rgb ye çevirme ve yazdırma
def show_bbox_coordinates(box, image_bgr, label):
    result_image = draw_bbox,(
        image_bgr,
        box,
        label
    )

    result_image_rgb = cv2.cvtColor(
        result_image,
        cv2.COLOR_BGR2RGB
    )

    plt.imshow(result_image_rgb)
    plt.axis("off")
    plt.show()

In [ ]:
# Uçtan uca test

user_question = "What is pixel bbox coordinates of cat?"

# 1. Prompt oluştur
question = build_prompt(user_question)

# 2. Modelden ham cevabı al
response = ask_model(
    processor,
    model,
    image_rgb,
    question
)
print("Modelin ham cevabı:")
print(response)
print("Response:", repr(response))


# 3. JSON stringi Python nesnesine çevir
detections = parse_model_response(response)

print("\nParse edilmiş detections:")
print(detections)

# 4. İlk nesnenin bilgilerini al
if detections:
    first_detection = detections[0]

    label = first_detection["label"]
    box = first_detection["box"]

    print("\nLabel:")
    print(label)

    print("\nModel bbox koordinatları:")
    print(box)

    # 5. Bbox çizilmiş resmi oluştur
    show_bbox_coordinates(box, image_bgr, label)

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Modelin ham cevabı:
[
102 213 343 846]
Model's answer is not JSON.

Parse edilmiş detections:
[]
